In [65]:
from omero import gateway
from getpass import getpass
from pprint import pprint
import numpy as np
import imagej
import scyjava

In [53]:
def connect(hostname, username, password):
    """
    Connect to an OMERO server
    :param hostname: Host name
    :param username: User
    :param password: Password
    :return: Connected BlitzGateway
    """
    conn = gateway.BlitzGateway(username, password,
                        host=hostname, secure=True, port=4063)
    conn.connect()
    conn.c.enableKeepAlive(60)
    return conn


def disconnect(conn):
    """
    Disconnect from an OMERO server
    :param conn: The BlitzGateway
    """
    conn.close()

class ChannelObject:
	def __init__(self, channel):
		self.channel = channel
		self.name = channel.getName()
		self.emission_wave = channel.getEmissionWave()
		self.excitation_wave = channel.getExcitationWave()
    
class ImageObject:
	def __init__(self, image, c=None, t=None, z=None):
		self.image = image
		self.id = image.getId()
		self.name = image.getName()
		self.size_x = image.getSizeX()
		self.size_y = image.getSizeY()
		self.size_z = image.getSizeZ()
		self.size_c = image.getSizeC()
		self.size_t = image.getSizeT()
		self.dim_order = "TCZYX"
		print("Loading image data...")
		self.image_data = self.getImageData(c, t, z)
		print("Image data loaded.")
		self.shape = self.image_data.shape
		self.objective = image.getObjectiveSettings()
		self.refractive_index = self.objective.getRefractiveIndex()
		self.NA = self.objective.getObjective().getLensNA()
		self.channels = [ChannelObject(ch) for ch in image.getChannels()]
	
	def getImageData(self, c, t, z):
		image_data = np.zeros((self.size_t, self.size_c, self.size_z, self.size_y, self.size_x))
		for z_plane in range(self.size_z):
			if z is not None:
				if z_plane not in z:
					continue
			for c_plane in range(self.size_c):
				if c is not None:
					if c_plane not in c:
						continue
				for t_plane in range(self.size_t):
					if t is not None:
						if t_plane not in t:
							continue
					image_data[t, c, z, :, :] = np.array(self.image.getPrimaryPixels().getPlane(z_plane, c_plane, t_plane))
		return image_data

In [ ]:
omero_hostname = r"fms-fac-omero.ncl.ac.uk"
omero_username = "njg135"
temp_password = getpass("OMERO Password: ")

In [30]:
connection = gateway.BlitzGateway(omero_username, temp_password, host=omero_hostname)
print ("Connecting to OMERO server...")
connection.connect()
print ("Connection successful!")

Connecting to OMERO server...
Connection successful!


In [66]:
ij = imagej.init(r"C:\Other_Program_Files\Fiji.app")

RuntimeError: Can't find org.jpype.jar support library

In [40]:
# ImageID = 126919 #Widefield
# ImageID = 126993 #Confocal
# ImageID = 125329 #CBCB
ImageID = 40460 #AxioTetra

In [41]:
img = connection.getObject("Image", ImageID)

In [47]:
objsettings = img.getObjectiveSettings()
objective = objsettings.getObjective()
NA = objective.getLensNA()

In [54]:
imp = ImageObject(img)

Loading image data...
Image data loaded.


In [63]:
imp.channels[3].name

'DAPI'

In [54]:
imp.shape

(1, 1, 51, 1024, 1024)

In [37]:
disconnect(connection)